In [1]:
'''
AA-CBR evaluation using a trained 2D slot attention model on BrainWear PNG slices.

Outcomes are EORTC PRO scores from eortc_scores.csv, binned into ordinal classes.
Sweeps all combinations of score_name, char model, n_bins, agg_mode, strategy, and strict.

Efficiency: slot model runs once per slice (Phase 0), char features are aggregated
per (char_model, agg_mode) without re-running the model (Phase 1a), and outcomes
are loaded per (score_name, n_bins) from the CSV directly (Phase 1b).

Set CHECKPOINT and SCORE_NAMES below before running.
'''

import csv as csv_mod
import sys
from itertools import product as iproduct
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

FYP_ROOT = Path('/path/to/BrainWear_Kareem/FYP')
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

from aacbr.aacbr_parallel import AACBRParallel
from aacbr.configs.brats_model_config import BraTSOutcomeConfig
from utils.characterisations import (
    TumourCharacterisationLarge2D,
    TumourCharacterisationSmall2D,
    TumourCharacterisationSmall2DV2,
    TumourCharacterisationLarge2DV2,
)
import torchvision.transforms.functional as TF

CHAR_MODELS = {
    'small':    TumourCharacterisationSmall2D,
    'large':    TumourCharacterisationLarge2D,
    'small_v2': TumourCharacterisationSmall2DV2,
    'large_v2': TumourCharacterisationLarge2DV2,
}

# All 26 available EORTC scores:
ALL_SCORES = [
    'QL2', 'PF2', 'RF2', 'EF', 'CF', 'SF', 'FA', 'NV', 'PA', 'DY',
    'SL', 'AP', 'CO', 'DI', 'FI', 'BNFU', 'BNVD', 'BNMD', 'BNCD',
    'BNHA', 'BNSE', 'BNDR', 'BNIS', 'BNHL', 'BNWL', 'BNBC',
]

DATA_DIR   = str(
    Path('/path/to/BrainWear_Kareem')
    / 'Processed_Brainwear_PNG_fixed_norm'
)
SCORE_FILE = str(
    Path('/path/to/BrainWear_Kareem')
    / 'eortc_scores.csv'
)
CONFIG     = str(FYP_ROOT / 'aacbr' / 'configs' / 'brats_configs' / 'flat_config.json')
DEFAULT_TRAINED_LEADERBOARD = str(FYP_ROOT / 'aacbr' / 'leaderboard_trained.json')

In [2]:
def load_slot_model(checkpoint_path: str, device: torch.device):
    from slot_attention.training_2d.slot_attention_2d import SlotClassifier2D

    ckpt = torch.load(checkpoint_path, map_location=device)
    hp = ckpt.get('hyperparameters', {})
    model = SlotClassifier2D(
        in_shape=hp.get('in_shape', (1, 240, 240)),
        width=hp.get('width', 64),
        num_slots=hp.get('num_slots', 5),
        slot_dim=hp.get('slot_dim', 64),
        routing_iters=hp.get('routing_iters', 7),
        temperature=hp.get('temperature', 0.5),
        encoder_depth=hp.get('encoder_depth', 4),
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device).eval()
    epoch = ckpt.get('epoch', '?')
    print(f'Loaded 2D slot model from {checkpoint_path}  (epoch {epoch})')
    return model


def load_eortc_scores(score_file: str) -> dict[str, dict[str, float]]:
    '''Parse eortc_scores.csv → {score_name: {pid: float_score}}.'''
    scores: dict[str, dict[str, float]] = {}
    with open(score_file, newline='', encoding='utf-8') as f:
        reader = csv_mod.DictReader(f)
        # fieldnames[0] is 'Question'; the rest are patient IDs
        q_col = reader.fieldnames[0]
        pids = [h.strip() for h in reader.fieldnames[1:] if h.strip()]
        for row in reader:
            name = row[q_col].strip()
            if not name or name == 'Date':
                continue
            vals: dict[str, float] = {}
            for pid in pids:
                try:
                    vals[pid] = float(row[pid].strip())
                except (ValueError, KeyError, TypeError):
                    pass
            if vals:
                scores[name] = vals
    print(f'Loaded {len(scores)} EORTC scores for up to {len(pids)} patients.')
    return scores


def get_outcomes(
    eortc_scores: dict[str, dict[str, float]],
    score_name: str,
    n_bins: int,
    available_pids: list[str],
) -> tuple[dict[str, int], np.ndarray]:
    '''Quantile-bin a given EORTC score for patients present in available_pids.'''
    score_dict = eortc_scores.get(score_name, {})
    valid_pids = [p for p in available_pids if p in score_dict]
    if not valid_pids:
        return {}, np.array([])

    raw = np.array([score_dict[p] for p in valid_pids])
    quantiles = np.quantile(raw, np.linspace(0, 1, n_bins + 1)[1:-1])
    bins = np.digitize(raw, quantiles)

    print(f'  {score_name}: {len(valid_pids)} patients, {n_bins} bins')
    print(f'  Score range  min={raw.min():.1f}  median={np.median(raw):.1f}  max={raw.max():.1f}')
    print(f'  Quantile thresholds: {np.round(quantiles, 1).tolist()}')
    print(f'  Class distribution: {np.bincount(bins, minlength=n_bins).tolist()}')
    return {p: int(b) for p, b in zip(valid_pids, bins)}, quantiles


def _find_axial_t2_dir(patient_dir: Path) -> Path | None:
    '''Return the first sorted Axial_T2* subdirectory, or None if absent.'''
    candidates = sorted(
        d for d in patient_dir.iterdir()
        if d.is_dir() and d.name.startswith('Axial_T2')
    )
    return candidates[0] if candidates else None


def extract_slot_outputs_2d(
    data_dir: str,
    slot_model,
    device: torch.device,
) -> dict[str, list[torch.Tensor]]:
    '''Run the 2D slot model on every PNG slice once. Returns {pid: [y_hat per slice]}.'''
    patient_dirs = sorted(p for p in Path(data_dir).iterdir() if p.is_dir())
    slot_cache: dict[str, list[torch.Tensor]] = {}

    with torch.no_grad():
        for patient_dir in tqdm(patient_dirs, desc='Extracting slot outputs (2D)'):
            pid = patient_dir.name
            axial_dir = _find_axial_t2_dir(patient_dir)
            if axial_dir is None:
                continue
            slices = sorted(axial_dir.glob('*.png'))
            if not slices:
                continue
            pid_outputs = []
            for slice_path in slices:
                t2 = TF.to_tensor(Image.open(slice_path).convert('L'))  # (1, H, W)
                _, _, _, _, y_hat = slot_model(t2.unsqueeze(0).to(device))
                pid_outputs.append(y_hat[0].cpu())
            slot_cache[pid] = pid_outputs

    print(f'Slot outputs cached for {len(slot_cache)} patients '
          f'({sum(len(v) for v in slot_cache.values())} slices total).')
    return slot_cache


def compute_char_features(
    slot_cache: dict[str, list[torch.Tensor]],
    char_model,
    agg_mode: str = 'sum',
) -> dict[str, np.ndarray]:
    '''Aggregate per-slice slot outputs into one feature vector per patient.'''
    features: dict[str, np.ndarray] = {}
    for pid, slice_outputs in slot_cache.items():
        acc = char_model.default_case().astype(np.int64)
        for slots in slice_outputs:
            v = char_model.characterisation_transform(slots).astype(np.int64)
            if agg_mode == 'max':
                acc = np.maximum(acc, v)
            else:
                acc = acc + v
        features[pid] = acc
    return features


def _dedup(feats: np.ndarray, out: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    unique, inverse = np.unique(feats, axis=0, return_inverse=True)
    dedup_out = np.array([
        int(np.round(np.median(out[inverse == i])))
        for i in range(len(unique))
    ])
    return unique, dedup_out


def run_aacbr(
    train_feats: np.ndarray,
    train_out: np.ndarray,
    test_feats: np.ndarray,
    test_out: np.ndarray,
    char_model,
    cfg: BraTSOutcomeConfig,
    n_bins: int,
    *,
    strategy: str = 'ordinal',
    strict: bool = True,
    use_supports: bool = False,
    smart_default: bool = False,
    dedup: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    ls_fn = lambda a, b: char_model.less_specific(a, b, strict=strict)

    def make_model(k):
        default_out = (1 if k == 0 else 0) if smart_default else cfg.default_outcome
        return AACBRParallel(
            less_specific=ls_fn,
            default_case=char_model.default_case(),
            default_outcome=default_out,
            include_supports=use_supports,
            supported_attack_chain=use_supports,
        )

    models = {k: make_model(k) for k in range(n_bins)}

    if strategy == 'ordinal':
        f, o = _dedup(train_feats, train_out) if dedup else (train_feats, train_out)
        for k in range(n_bins):
            models[k].fit(f, (o >= k).astype(int))
        binary = np.stack([models[k].predict(test_feats) for k in range(n_bins)], axis=1)
        preds = binary[:, 1:].sum(axis=1)

    elif strategy == 'flat':
        f, o = _dedup(train_feats, train_out) if dedup else (train_feats, train_out)
        for k in range(n_bins):
            models[k].fit(f, (o == k).astype(int))
        binary = np.stack([models[k].predict(test_feats) for k in range(n_bins)], axis=1)
        preds = np.array([
            max(np.flatnonzero(binary[i] == 1), default=cfg.default_class)
            for i in range(len(test_feats))
        ])

    elif strategy == 'tournament_flat':
        for k in range(n_bins - 1):
            mask = train_out >= k
            feats_k = train_feats[mask]
            out_k = (train_out[mask] == k).astype(int)
            if dedup:
                feats_k, out_k = _dedup(feats_k, out_k)
            models[k].fit(feats_k, out_k)
        preds = np.full(len(test_feats), n_bins - 1, dtype=int)
        unclassified = np.ones(len(test_feats), dtype=bool)
        for k in range(n_bins - 1):
            preds_k = models[k].predict(test_feats[unclassified])
            newly = np.zeros(len(test_feats), dtype=bool)
            newly[unclassified] = preds_k == 1
            preds[newly] = k
            unclassified[newly] = False

    elif strategy == 'tournament_tree':
        for k in range(n_bins - 1):
            mask = train_out >= k
            feats_k = train_feats[mask]
            out_k = (train_out[mask] > k).astype(int)
            if dedup:
                feats_k, out_k = _dedup(feats_k, out_k)
            models[k].fit(feats_k, out_k)
        preds = np.full(len(test_feats), n_bins - 1, dtype=int)
        at_node = np.ones(len(test_feats), dtype=bool)
        for k in range(n_bins - 1):
            preds_k = models[k].predict(test_feats[at_node])
            left_mask = np.zeros(len(test_feats), dtype=bool)
            left_mask[at_node] = preds_k == 0
            preds[left_mask] = k
            at_node[left_mask] = False

    else:
        raise ValueError(f'Unknown strategy: {strategy!r}')

    return test_out, preds


def report(tag: str, y_true: np.ndarray, y_pred: np.ndarray, n_bins: int) -> dict:
    from sklearn.metrics import (
        accuracy_score, balanced_accuracy_score,
        confusion_matrix, classification_report,
    )
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mae     = np.abs(y_true.astype(float) - y_pred.astype(float)).mean()
    labels  = list(range(n_bins))
    cm      = confusion_matrix(y_true, y_pred, labels=labels)
    report_dict = classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=3, output_dict=True)
    macro = report_dict.get('macro avg', {})
    print(f'\n=== {tag} ===')
    print(f'Accuracy: {acc:.4f}   Balanced Acc: {bal_acc:.4f}   Ordinal MAE: {mae:.4f}')
    print('Confusion matrix (rows=true, cols=pred):')
    print(cm)
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=3))
    return {
        'accuracy':          float(acc),
        'balanced_accuracy': float(bal_acc),
        'f1':                float(macro.get('f1-score', 0.0)),
        'precision':         float(macro.get('precision', 0.0)),
        'recall':            float(macro.get('recall', 0.0)),
        'ordinal_mae':       float(mae),
        'confusion_matrix':  cm.tolist(),
        'per_class': {
            k: {m: v for m, v in vs.items() if m != 'support'}
            for k, vs in report_dict.items()
            if isinstance(vs, dict)
        },
    }

In [6]:
# ── User configuration ─────────────────────────────────────────────────────────
CHECKPOINT  = '/path/to/BrainWear_Kareem/FYP/slot_attention/training_2d/models/checkpoints/brats_png_v14a_0.15_test/ckpt.pt'   # <-- set to your .pt file

SEED        = 0
TRAIN_FRAC  = 0.8
NUM_SLOTS   = 5
AGG_MODES   = ['sum', 'max']
STRATEGIES  = ['ordinal', 'flat', 'tournament_flat', 'tournament_tree']
N_BINS      = [2, 3, 4, 5]

# Select which EORTC scores to evaluate. Use ALL_SCORES for a full sweep.
SCORE_NAMES = ['QL2', 'PF2', 'CF', 'EF']
# SCORE_NAMES = ALL_SCORES  # uncomment for all 26 scores
# ───────────────────────────────────────────────────────────────────────────────

np.random.seed(SEED)
torch.manual_seed(SEED)

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg        = BraTSOutcomeConfig.from_json(CONFIG)
slot_model = load_slot_model(CHECKPOINT, device)
print(f'Device: {device}')

# -------------------------------------------------------------------
# Phase 0: Load EORTC scores from CSV (no model needed)
# -------------------------------------------------------------------
eortc_scores = load_eortc_scores(SCORE_FILE)

# -------------------------------------------------------------------
# Phase 1a: Extract slot outputs ONCE for all patients / all slices
# -------------------------------------------------------------------
slot_cache = extract_slot_outputs_2d(DATA_DIR, slot_model, device)
all_pids   = list(slot_cache.keys())

# -------------------------------------------------------------------
# Phase 1b: Compute char features per (char_model, agg_mode)
#           No further GPU work -- reuses slot_cache.
# -------------------------------------------------------------------
print('\nComputing char features per (char_model, agg_mode)...')
char_feat_cache: dict[tuple, dict[str, np.ndarray]] = {}
for char_name, agg_mode in iproduct(CHAR_MODELS.keys(), AGG_MODES):
    char_model = CHAR_MODELS[char_name]
    print(f'  char={char_name} ({char_model.DIMS}D), agg={agg_mode}')
    char_feat_cache[(char_name, agg_mode)] = compute_char_features(slot_cache, char_model, agg_mode)

# -------------------------------------------------------------------
# Phase 1c: Build feature_cache per (score_name, char_name, n_bins, agg_mode)
# -------------------------------------------------------------------
print('\nBuilding feature cache (outcomes + train/test splits)...')
feature_cache: dict[tuple, tuple] = {}

for score_name, char_name, n_bins, agg_mode in iproduct(
    SCORE_NAMES, CHAR_MODELS.keys(), N_BINS, AGG_MODES
):
    pid_to_class, _ = get_outcomes(eortc_scores, score_name, n_bins, all_pids)
    if not pid_to_class:
        continue

    pid_feat = char_feat_cache[(char_name, agg_mode)]
    matched  = [p for p in pid_feat if p in pid_to_class]
    if len(matched) < 4:
        continue

    features = np.array([pid_feat[p] for p in matched])
    outcomes = np.array([pid_to_class[p] for p in matched])

    idx          = np.arange(len(matched))
    class_counts = np.bincount(outcomes, minlength=n_bins)
    if np.any(class_counts < 2):
        rng     = np.random.default_rng(SEED)
        perm    = rng.permutation(len(idx))
        n_train = int(round(TRAIN_FRAC * len(idx)))
        train_idx, test_idx = perm[:n_train], perm[n_train:]
    else:
        train_idx, test_idx = train_test_split(
            idx, train_size=TRAIN_FRAC, random_state=SEED, stratify=outcomes
        )
    feature_cache[(score_name, char_name, n_bins, agg_mode)] = (
        features, outcomes, train_idx, test_idx
    )

print(f'Feature cache: {len(feature_cache)} combinations.')

# -------------------------------------------------------------------
# Phase 2: Sweep all strategy x strict combinations
# -------------------------------------------------------------------
n_combos = len(feature_cache) * len(STRATEGIES) * 2
print(f'\nRunning AA-CBR sweep ({n_combos} configurations)...')
sweep_results = []

for (score_name, char_name, n_bins, agg_mode), strategy, strict in iproduct(
    feature_cache.keys(), STRATEGIES, [True, False]
):
    features, outcomes, train_idx, test_idx = feature_cache[(score_name, char_name, n_bins, agg_mode)]
    char_model = CHAR_MODELS[char_name]

    y_true, y_pred = run_aacbr(
        features[train_idx], outcomes[train_idx],
        features[test_idx],  outcomes[test_idx],
        char_model, cfg, n_bins,
        strategy=strategy, strict=strict, use_supports=False,
        smart_default=False, dedup=True,
    )
    acc = float(accuracy_score(y_true, y_pred))
    mae = float(np.abs(y_true.astype(float) - y_pred.astype(float)).mean())
    sweep_results.append(dict(
        score=score_name, char=char_name, n_bins=n_bins, agg=agg_mode,
        strategy=strategy, strict=strict,
        acc=acc, mae=mae, y_true=y_true, y_pred=y_pred,
    ))

sweep_results.sort(key=lambda r: (-r['acc'], r['mae']))

# -------------------------------------------------------------------
# Phase 3: Summary table + full report for the best overall config
# -------------------------------------------------------------------
print(f"\n{'score':<8} {'char':<10} {'agg':<5} {'bins':>4} {'strategy':<16} {'strict':>6} {'Acc':>8} {'MAE':>8}")
print('-' * 79)
for r in sweep_results:
    print(
        f"{r['score']:<8} {r['char']:<10} {r['agg']:<5} {r['n_bins']:>4} "
        f"{r['strategy']:<16} {str(r['strict']):>6} {r['acc']:>8.4f} {r['mae']:>8.4f}"
    )

best = sweep_results[0]
metrics = report(
    f"BEST -- score={best['score']}, char={best['char']}, agg={best['agg']}, "
    f"n_bins={best['n_bins']}, strategy={best['strategy']}, strict={best['strict']}",
    best['y_true'], best['y_pred'], best['n_bins'],
)

In [7]:
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support

print('k-fold stratified CV (adaptive k) -- best feasible config per strategy x score x n_bins\n')
print(f"{'strategy':<16} {'score':<8} {'bins':>4} {'char':<10} {'agg':<5} {'strict':>6} {'n':>4} {'k':>2}   {'Acc':>6} +/- {'std':>6}   {'BalAcc':>7} +/- {'std':>6}   {'F1':>6} +/- {'std':>6}")
print('-' * 130)

cv_results: dict[tuple, dict] = {}  # (strategy, score_name, n_bins) -> CV stats + config

for strategy, score_name, n_bins in iproduct(STRATEGIES, SCORE_NAMES, N_BINS):
    candidates = [
        r for r in sweep_results
        if r['strategy'] == strategy and r['score'] == score_name and r['n_bins'] == n_bins
    ]
    if not candidates:
        continue

    # Pick the highest-accuracy config where CV is feasible (all bins non-empty, >=5 patients).
    b, features_b, outcomes_b = None, None, None
    for cand in candidates:
        key_cand = (cand['score'], cand['char'], cand['n_bins'], cand['agg'])
        if key_cand not in feature_cache:
            continue
        feats_c, outs_c, _, _ = feature_cache[key_cand]
        cc = np.bincount(outs_c, minlength=cand['n_bins'])
        if cc.min() > 0 and len(outs_c) >= 5:
            b, features_b, outcomes_b = cand, feats_c, outs_c
            break

    if b is None:
        continue  # skip silently -- too many combinations to print for all scores x bins

    char_model_b = CHAR_MODELS[b['char']]
    min_class    = int(np.bincount(outcomes_b, minlength=b['n_bins']).min())
    n_splits_eff = min(5, min_class)
    kf_eff       = StratifiedKFold(n_splits=n_splits_eff, shuffle=True, random_state=SEED)

    fold_accs, fold_maes, fold_bal_accs, fold_f1s, fold_precisions, fold_recalls = [], [], [], [], [], []
    for train_idx, test_idx in kf_eff.split(features_b, outcomes_b):
        y_true, y_pred = run_aacbr(
            features_b[train_idx], outcomes_b[train_idx],
            features_b[test_idx],  outcomes_b[test_idx],
            char_model_b, cfg, b['n_bins'],
            strategy=strategy, strict=b['strict'],
            use_supports=False, smart_default=False, dedup=True,
        )
        all_labels = list(range(b['n_bins']))
        fold_accs.append(float(accuracy_score(y_true, y_pred)))
        fold_maes.append(float(np.abs(y_true.astype(float) - y_pred.astype(float)).mean()))
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message='y_pred contains classes not in y_true')
            fold_bal_accs.append(float(balanced_accuracy_score(y_true, y_pred)))
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0, labels=all_labels)
        fold_f1s.append(float(f))
        fold_precisions.append(float(p))
        fold_recalls.append(float(r))

    cv_results[(strategy, score_name, n_bins)] = dict(
        b=b, features_b=features_b, outcomes_b=outcomes_b, n_splits=n_splits_eff,
        fold_accs=fold_accs, fold_maes=fold_maes,
        fold_bal_accs=fold_bal_accs, fold_f1s=fold_f1s,
        fold_precisions=fold_precisions, fold_recalls=fold_recalls,
    )

    print(
        f"{strategy:<16} {score_name:<8} {b['n_bins']:>4} {b['char']:<10} {b['agg']:<5} "
        f"{str(b['strict']):>6} {len(outcomes_b):>4} {n_splits_eff:>2}   "
        f"{np.mean(fold_accs):>6.4f} +/- {np.std(fold_accs):>6.4f}   "
        f"{np.mean(fold_bal_accs):>7.4f} +/- {np.std(fold_bal_accs):>6.4f}   "
        f"{np.mean(fold_f1s):>6.4f} +/- {np.std(fold_f1s):>6.4f}"
    )

In [8]:
# ── log to trained-model leaderboard (set SAVE_RESULTS = True to persist) ─────
# Reads from cv_results populated by the CV cell -- no recomputation.
SAVE_RESULTS = True
NOTES = ''

if SAVE_RESULTS:
    from aacbr.results_logger import log_result
    for (strategy, score_name, n_bins), res in cv_results.items():
        b               = res['b']
        features_b      = res['features_b']
        outcomes_b      = res['outcomes_b']
        fold_accs       = res['fold_accs']
        fold_maes       = res['fold_maes']
        fold_bal_accs   = res['fold_bal_accs']
        fold_f1s        = res['fold_f1s']
        fold_precisions = res['fold_precisions']
        fold_recalls    = res['fold_recalls']

        char_model_b = CHAR_MODELS[b['char']]
        kf_eff = StratifiedKFold(n_splits=res['n_splits'], shuffle=True, random_state=SEED)

        # Collect OOF predictions for confusion matrix / per-class metrics
        oof_true, oof_pred = [], []
        for train_idx, test_idx in kf_eff.split(features_b, outcomes_b):
            yt, yp = run_aacbr(
                features_b[train_idx], outcomes_b[train_idx],
                features_b[test_idx],  outcomes_b[test_idx],
                char_model_b, cfg, b['n_bins'],
                strategy=b['strategy'], strict=b['strict'],
                use_supports=False, smart_default=False, dedup=True,
            )
            oof_true.extend(yt)
            oof_pred.extend(yp)

        oof_true = np.array(oof_true)
        oof_pred = np.array(oof_pred)

        oof_metrics = report(
            f"strategy={strategy} score={score_name} n_bins={n_bins} "
            f"[char={b['char']}, agg={b['agg']}, strict={b['strict']}]",
            oof_true, oof_pred, b['n_bins'],
        )

        log_result(
            {
                'eval_script': 'eval_trained_brainwear_2d',
                'config': {
                    'checkpoint': CHECKPOINT,
                    'score_name': score_name,
                    'char_model': b['char'],
                    'n_bins': b['n_bins'],
                    'agg_mode': b['agg'],
                    'num_slots': NUM_SLOTS,
                    'strategy': b['strategy'],
                    'strict': b['strict'],
                    'n_folds': res['n_splits'],
                    'seed': SEED,
                },
                'data_stats': {
                    'n_patients': int(len(outcomes_b)),
                    'class_distribution': np.bincount(outcomes_b, minlength=b['n_bins']).tolist(),
                },
                'metrics': {
                    'cv_mean_accuracy':          float(np.mean(fold_accs)),
                    'cv_std_accuracy':           float(np.std(fold_accs)),
                    'cv_mean_balanced_accuracy': float(np.mean(fold_bal_accs)),
                    'cv_std_balanced_accuracy':  float(np.std(fold_bal_accs)),
                    'cv_mean_f1':                float(np.mean(fold_f1s)),
                    'cv_std_f1':                 float(np.std(fold_f1s)),
                    'cv_mean_precision':         float(np.mean(fold_precisions)),
                    'cv_std_precision':          float(np.std(fold_precisions)),
                    'cv_mean_recall':            float(np.mean(fold_recalls)),
                    'cv_std_recall':             float(np.std(fold_recalls)),
                    'cv_mean_mae':               float(np.mean(fold_maes)),
                    'cv_std_mae':                float(np.std(fold_maes)),
                    'fold_accs':                 fold_accs,
                    'fold_maes':                 fold_maes,
                    'fold_balanced_accs':        fold_bal_accs,
                    'fold_f1s':                  fold_f1s,
                    'oof_accuracy':              oof_metrics['accuracy'],
                    'oof_balanced_accuracy':     oof_metrics['balanced_accuracy'],
                    'oof_f1':                    oof_metrics['f1'],
                    'oof_precision':             oof_metrics['precision'],
                    'oof_recall':                oof_metrics['recall'],
                    'oof_ordinal_mae':           oof_metrics['ordinal_mae'],
                    'confusion_matrix':          oof_metrics['confusion_matrix'],
                    'per_class':                 oof_metrics['per_class'],
                },
                'notes': NOTES,
            },
            leaderboard_path=DEFAULT_TRAINED_LEADERBOARD,
        )
    print(f'Logged {len(cv_results)} entries to leaderboard.')